# Ensure lakeLoom Service Principal

Creates (or finds) the least-privilege service principal for ZeroBus ingestion
and establishes the secret-scope credential contract used by the Databricks App
and iOS QR-pair onboarding flow.

This notebook runs as the **first task** in the `platform_bootstrap` job. It
outputs the SPN `application_id` as a **task value** so the downstream DDL task
can grant table-level permissions.

**Secret scope keys auto-provisioned by this notebook:**

| Key | Source | Purpose |
| --- | --- | --- |
| `{client_id_dbs_key}` | SPN `application_id` | OAuth M2M client identifier |
| `workspace_url` | Derived from config | Databricks workspace URL |
| `zerobus_endpoint` | Workspace ID + region | ZeroBus gRPC endpoint |
| `target_table_name` | Catalog + schema | Fully qualified bronze table |
| `zerobus_stream_pool_size` | Job parameter | SDK gRPC stream pool size |

**Admin-provisioned (manual after first deploy):**

| Key | Source | Purpose |
| --- | --- | --- |
| `{client_secret_dbs_key}` | Workspace UI or CLI | OAuth M2M client secret |

In [0]:
%pip install --upgrade databricks-sdk
dbutils.library.restartPython()

In [0]:
dbutils.widgets.text("catalog_use", "")
dbutils.widgets.text("schema_use", "")
dbutils.widgets.text("secret_scope_name", "")
dbutils.widgets.text("client_id_dbs_key", "")
dbutils.widgets.text("client_secret_dbs_key", "")
dbutils.widgets.text("zerobus_stream_pool_size", "")

catalog_use = dbutils.widgets.get("catalog_use")
schema_use = dbutils.widgets.get("schema_use")
secret_scope_name = dbutils.widgets.get("secret_scope_name")
client_id_dbs_key = dbutils.widgets.get("client_id_dbs_key")
client_secret_dbs_key = dbutils.widgets.get("client_secret_dbs_key")
zerobus_stream_pool_size = dbutils.widgets.get("zerobus_stream_pool_size")

print(f"Catalog:              {catalog_use}")
print(f"Schema:               {schema_use}")
print(f"Secret scope:         {secret_scope_name}")
print(f"Client ID key:        {client_id_dbs_key}")
print(f"Client secret key:    {client_secret_dbs_key}")
print(f"Stream pool size:     {zerobus_stream_pool_size}")

In [0]:
import sys
from pathlib import Path

# Add src/lib to the import path (relative to this notebook)
notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
lib_path = str(Path("/Workspace") / Path(notebook_path).parent.relative_to("/") / ".." / "lib")
if lib_path not in sys.path:
    sys.path.insert(0, lib_path)

from workspace_metadata import get_workspace_id, get_region, get_zerobus_endpoint
from service_principal import get_or_create_service_principal, verify_client_credentials
from secrets import put_secret, list_secret_keys, ensure_scope_read_acl, try_get_secret_value

print(f"Loaded lib from: {lib_path}")

In [0]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
workspace_url = w.config.host.rstrip("/")

workspace_id = get_workspace_id(w)
if not workspace_id:
    raise RuntimeError(
        "Could not determine workspace ID. "
        "Set DATABRICKS_WORKSPACE_ID or run on supported Databricks compute."
    )

region = get_region(workspace_url)
zerobus_endpoint = get_zerobus_endpoint(workspace_id, region)
target_table_name = f"{catalog_use}.{schema_use}.transcript_events_raw"

print(f"Workspace URL:        {workspace_url}")
print(f"Workspace ID:         {workspace_id}")
print(f"AWS region:           {region}")
print(f"ZeroBus endpoint:     {zerobus_endpoint}")
print(f"Target table:         {target_table_name}")

In [0]:
spn_display_name = f"lakeloom-{schema_use}"
print(f"Target SPN display name: {spn_display_name}")

spn, is_new_spn = get_or_create_service_principal(w, spn_display_name)
spn_application_id = spn.application_id

print(f"Application ID:       {spn_application_id}")
print(f"Workspace object ID:  {spn.id}")
print(f"Created this run:     {is_new_spn}")

In [0]:
secrets_to_store = {
    client_id_dbs_key: spn_application_id,
    "workspace_url": workspace_url,
    "zerobus_endpoint": zerobus_endpoint,
    "target_table_name": target_table_name,
    "zerobus_stream_pool_size": zerobus_stream_pool_size,
}

for key, value in secrets_to_store.items():
    put_secret(w, secret_scope_name, key, value)
    print(f"Stored {key} = {value}")

# Check for admin-provisioned client_secret
existing_keys = list_secret_keys(w, secret_scope_name)
credentials_provisioned = client_secret_dbs_key in existing_keys
print(f"\nAvailable keys:       {sorted(existing_keys)}")
print(f"Client secret present: {'YES' if credentials_provisioned else 'NO — admin action required'}")

In [0]:
# Grant the SPN READ access to the secret scope so the Databricks App
# can retrieve credentials at runtime. put_acl is idempotent.
ensure_scope_read_acl(w, secret_scope_name, spn_application_id)
print(f"Granted READ on scope '{secret_scope_name}' to {spn_application_id}")

In [0]:
m2m_token_verified = False
m2m_verification_status = "skipped"
verification_details = "client_secret not available to this run"

if credentials_provisioned:
    client_secret_value, read_error = try_get_secret_value(
        secret_scope_name, client_secret_dbs_key
    )
    if client_secret_value:
        ok, status_code, preview = verify_client_credentials(
            workspace_url=workspace_url,
            client_id=spn_application_id,
            client_secret=client_secret_value,
        )
        m2m_token_verified = ok
        m2m_verification_status = f"http_{status_code}"
        verification_details = preview or "token response had empty body"
        if not ok:
            raise RuntimeError(
                f"OAuth client credentials verification failed "
                f"with status {status_code}: {preview}"
            )
        print(f"M2M token verification: PASSED (HTTP {status_code})")
    else:
        m2m_verification_status = "skipped"
        verification_details = (
            f"client_secret exists but could not be read: {read_error}"
        )
        print(f"M2M token verification: SKIPPED — {verification_details}")
else:
    print("M2M token verification: SKIPPED — client_secret not yet provisioned")

In [0]:
# The DDL task reads this via:
#   {{tasks.ensure_service_principal.values.spn_application_id}}
dbutils.jobs.taskValues.set(key="spn_application_id", value=spn_application_id)
print(f"Task value set: spn_application_id = {spn_application_id}")

In [0]:
import json

summary = {
    "spn_display_name": spn_display_name,
    "spn_application_id": spn_application_id,
    "spn_workspace_object_id": spn.id,
    "created_this_run": is_new_spn,
    "workspace_url": workspace_url,
    "workspace_id": workspace_id,
    "aws_region": region,
    "zerobus_endpoint": zerobus_endpoint,
    "target_table_name": target_table_name,
    "secret_scope_name": secret_scope_name,
    "client_id_dbs_key": client_id_dbs_key,
    "client_secret_dbs_key": client_secret_dbs_key,
    "client_secret_present": credentials_provisioned,
    "m2m_token_verified": m2m_token_verified,
    "m2m_verification_status": m2m_verification_status,
    "next_manual_step": (
        f"Generate and store {client_secret_dbs_key} in secret scope "
        f"{secret_scope_name} for service principal {spn_application_id} "
        f"if it is not already present."
    ),
}

print("=" * 70)
print("Platform Bootstrap — Service Principal Summary")
print("=" * 70)
print(json.dumps(summary, indent=2, sort_keys=True))